<a href="https://colab.research.google.com/github/amanpoonia/Research_project_Nour/blob/main/Tables_transition_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code is structured in these Steps:
1) Choose the desired asset
available choices: EURUSD, S&P500, BITCOIN-USD, BRENT CRUDE, GOLD
2) Choose how many steps back in past to consider.

  PART 1 => 1 step back

  PART 2 => 2 steps back


---



**PART 1: Returns at t conditional on t-1**

`(t-1,t)`


**PART 2: Returns at t conditional on t-1 and t-2**

`(t-2,t-1,t)`



---



*EURUSD*

# PART 1

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC']
ticker = ['EURUSD=X']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['EURUSD=X'] < lower['EURUSD=X'],
    log_returns_categorized['EURUSD=X'] > upper['EURUSD=X']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
'''
    (-1,-1) -> 0
    (-1,0) -> 1
    (-1,1) -> 2
    (0,-1) -> 3
    (0,0) -> 4
    (0,1) -> 5
    (1,-1) -> 6
    (1,0) -> 7
    (1,1) -> 8
'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#3X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(3, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=[-1, 0, 1],   # previous state
    columns=[-1, 0, 1]  # current state
)

print(round(transition_table,2))

#3X3 RETURNS MATRIX
sums = df.groupby('pattern')['EURUSD=X'].mean().reindex(range(9), fill_value=0)
sums *= 100
matrix = sums.values.reshape(3, 3)
table_3x3 = pd.DataFrame(
    matrix,
    index=[-1, 0, 1],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_3x3,3))

#PART 2

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','GSPC']
ticker = ['EURUSD=X']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['EURUSD=X'] < lower['EURUSD=X'],
    log_returns_categorized['EURUSD=X'] > upper['EURUSD=X']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
#This is different than lag of just 1, as there are two lags considered, total number of cases would be 27.
'''
    (-1,-1,-1) -> 0
    (-1,-1,0) -> 1
    (-1,-1,1) -> 2
    (-1,0,-1) -> 3
    (-1,0,0) -> 4
    (-1,0,1) -> 5
    (-1,1,-1) -> 6
    (-1,1,0) -> 7
    (-1,1,1) -> 8
    (0,-1,-1) -> 9
    (0,-1,0) -> 10
    (0,-1,1) -> 11
    (0,0,-1) -> 12
    (0,0,0) -> 13
    (0,0,1) -> 14
    (0,1,-1) -> 15
    (0,1,0) -> 16
    (0,1,1) -> 17
    (1,-1,-1) -> 18
    (1,-1,0) -> 19
    (1,-1,1) -> 20
    (1,0,-1) -> 21
    (1,0,0) -> 22
    (1,0,1) -> 23
    (1,1,-1) -> 24
    (1,1,0) -> 25
    (1,1,1) -> 26

'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(2) + 1) * 9 +
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#9X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(9, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous state
    columns=[-1, 0, 1]  # current state
)
print(round(transition_table,2))

#9X3 RETURNS MATRIX
sums = df.groupby('pattern')['EURUSD=X'].mean().reindex(range(27), fill_value=0)
sums *= 100
matrix = sums.values.reshape(9, 3)
table_9x3 = pd.DataFrame(
    matrix,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_9x3,3))

/tmp/ipykernel_10924/1988236133.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

          -1      0      1
-1,-1  26.29  38.73  34.98
-1,0   31.05  39.65  29.30
-1,1   32.31  42.33  25.36
0,-1   29.47  42.11  28.42
0,0    27.06  42.93  30.01
0,1    31.04  41.28  27.68
1,-1   29.86  33.74  36.40
1,0    28.98  39.87  31.16
1,1    36.68  36.18  27.14
          -1      0      1
-1,-1 -0.712  0.007  0.687
-1,0  -0.655  0.004  0.635
-1,1  -0.842  0.012  0.770
0,-1  -0.645  0.008  0.706
0,0   -0.602  0.001  0.588
0,1   -0.660 -0.005  0.624
1,-1  -0.751  0.005  0.734
1,0   -0.601  0.002  0.648
1,1   -0.717  0.009  0.682




---



S&P500(^GSPC)

#PART 1

In [5]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC']
ticker = ['^GSPC']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['^GSPC'] < lower['^GSPC'],
    log_returns_categorized['^GSPC'] > upper['^GSPC']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
'''
    (-1,-1) -> 0
    (-1,0) -> 1
    (-1,1) -> 2
    (0,-1) -> 3
    (0,0) -> 4
    (0,1) -> 5
    (1,-1) -> 6
    (1,0) -> 7
    (1,1) -> 8
'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#3X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(3, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=[-1, 0, 1],   # previous state
    columns=[-1, 0, 1]  # current state
)

print(round(transition_table,2))

#3X3 RETURNS MATRIX
sums = df.groupby('pattern')['^GSPC'].mean().reindex(range(9), fill_value=0)
sums *= 100
matrix = sums.values.reshape(3, 3)
table_3x3 = pd.DataFrame(
    matrix,
    index=[-1, 0, 1],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_3x3,3))

/tmp/ipykernel_10924/1803695666.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

       -1      0      1
-1  30.67  31.37  37.96
 0  29.36  43.78  26.86
 1  30.12  43.65  26.23
       -1      0      1
-1 -1.373  0.089  1.372
 0 -0.957  0.061  0.959
 1 -1.291  0.071  1.201


#PART 2

In [6]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC']
ticker = ['^GSPC']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['^GSPC'] < lower['^GSPC'],
    log_returns_categorized['^GSPC'] > upper['^GSPC']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
#This is different than lag of just 1, as there are two lags considered, total number of cases would be 27.
'''
    (-1,-1,-1) -> 0
    (-1,-1,0) -> 1
    (-1,-1,1) -> 2
    (-1,0,-1) -> 3
    (-1,0,0) -> 4
    (-1,0,1) -> 5
    (-1,1,-1) -> 6
    (-1,1,0) -> 7
    (-1,1,1) -> 8
    (0,-1,-1) -> 9
    (0,-1,0) -> 10
    (0,-1,1) -> 11
    (0,0,-1) -> 12
    (0,0,0) -> 13
    (0,0,1) -> 14
    (0,1,-1) -> 15
    (0,1,0) -> 16
    (0,1,1) -> 17
    (1,-1,-1) -> 18
    (1,-1,0) -> 19
    (1,-1,1) -> 20
    (1,0,-1) -> 21
    (1,0,0) -> 22
    (1,0,1) -> 23
    (1,1,-1) -> 24
    (1,1,0) -> 25
    (1,1,1) -> 26

'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(2) + 1) * 9 +
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#9X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(9, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous state
    columns=[-1, 0, 1]  # current state
)
print(round(transition_table,2))

#9X3 RETURNS MATRIX
sums = df.groupby('pattern')['^GSPC'].mean().reindex(range(27), fill_value=0)
sums *= 100
matrix = sums.values.reshape(9, 3)
table_9x3 = pd.DataFrame(
    matrix,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_9x3,3))

/tmp/ipykernel_10924/1592207971.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

          -1      0      1
-1,-1  30.32  31.22  38.46
-1,0   34.29  30.31  35.40
-1,1   35.10  32.18  32.72
0,-1   28.01  34.40  37.59
0,0    27.94  48.51  23.54
0,1    25.00  56.01  18.99
1,-1   34.56  27.42  38.02
1,0    27.71  47.13  25.16
1,1    29.89  43.39  26.72
          -1      0      1
-1,-1 -1.574  0.096  1.730
-1,0  -1.102  0.052  1.104
-1,1  -1.475  0.060  1.302
0,-1  -1.090  0.083  1.064
0,0   -0.810  0.062  0.829
0,1   -1.024  0.091  0.976
1,-1  -1.492  0.092  1.398
1,0   -1.028  0.064  0.975
1,1   -1.284  0.047  1.239




---



*BTC-USD*

#PART 1

In [7]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC', 'BTC-USD']
ticker = ['BTC-USD']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['BTC-USD'] < lower['BTC-USD'],
    log_returns_categorized['BTC-USD'] > upper['BTC-USD']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
'''
    (-1,-1) -> 0
    (-1,0) -> 1
    (-1,1) -> 2
    (0,-1) -> 3
    (0,0) -> 4
    (0,1) -> 5
    (1,-1) -> 6
    (1,0) -> 7
    (1,1) -> 8
'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#3X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(3, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=[-1, 0, 1],   # previous state
    columns=[-1, 0, 1]  # current state
)

print(round(transition_table,2))

#3X3 RETURNS MATRIX
sums = df.groupby('pattern')['BTC-USD'].mean().reindex(range(9), fill_value=0)
sums *= 100
matrix = sums.values.reshape(3, 3)
table_3x3 = pd.DataFrame(
    matrix,
    index=[-1, 0, 1],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_3x3,3))

/tmp/ipykernel_10924/1878605147.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

       -1      0      1
-1  31.37  34.84  33.79
 0  26.03  48.18  25.79
 1  33.90  34.22  31.88
       -1      0      1
-1 -3.962  0.224  3.688
 0 -3.170  0.140  3.292
 1 -3.142  0.051  4.004


#PART 2

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC','BTC-USD']
ticker = ['BTC-USD']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['BTC-USD'] < lower['BTC-USD'],
    log_returns_categorized['BTC-USD'] > upper['BTC-USD']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
#This is different than lag of just 1, as there are two lags considered, total number of cases would be 27.
'''
    (-1,-1,-1) -> 0
    (-1,-1,0) -> 1
    (-1,-1,1) -> 2
    (-1,0,-1) -> 3
    (-1,0,0) -> 4
    (-1,0,1) -> 5
    (-1,1,-1) -> 6
    (-1,1,0) -> 7
    (-1,1,1) -> 8
    (0,-1,-1) -> 9
    (0,-1,0) -> 10
    (0,-1,1) -> 11
    (0,0,-1) -> 12
    (0,0,0) -> 13
    (0,0,1) -> 14
    (0,1,-1) -> 15
    (0,1,0) -> 16
    (0,1,1) -> 17
    (1,-1,-1) -> 18
    (1,-1,0) -> 19
    (1,-1,1) -> 20
    (1,0,-1) -> 21
    (1,0,0) -> 22
    (1,0,1) -> 23
    (1,1,-1) -> 24
    (1,1,0) -> 25
    (1,1,1) -> 26

'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(2) + 1) * 9 +
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#9X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(9, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous state
    columns=[-1, 0, 1]  # current state
)
print(round(transition_table,2))

#9X3 RETURNS MATRIX
sums = df.groupby('pattern')['BTC-USD'].mean().reindex(range(27), fill_value=0)
sums *= 100
matrix = sums.values.reshape(9, 3)
table_9x3 = pd.DataFrame(
    matrix,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_9x3,3))

/tmp/ipykernel_10924/1440102131.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

          -1      0      1
-1,-1  31.96  30.93  37.11
-1,0   27.61  46.17  26.22
-1,1   33.97  34.93  31.10
0,-1   31.24  38.23  30.54
0,0    24.06  52.52  23.43
0,1    32.31  36.08  31.60
1,-1   30.79  35.08  34.13
1,0    28.13  42.08  29.79
1,1    35.53  31.47  32.99
          -1      0      1
-1,-1 -4.359  0.275  3.811
-1,0  -3.991  0.220  3.478
-1,1  -3.443  0.062  3.664
0,-1  -3.496  0.135  3.516
0,0   -2.947  0.131  3.117
0,1   -2.586  0.015  3.859
1,-1  -4.041  0.283  3.722
1,0   -2.708  0.073  3.385
1,1   -3.382  0.081  4.494




---



*BZ=F*

BRENT CRUDE OIL LAST DAY FINANCE

#PART 1

In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC', 'BTC-USD','BZ=F']
ticker = ['BZ=F']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['BZ=F'] < lower['BZ=F'],
    log_returns_categorized['BZ=F'] > upper['BZ=F']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
'''
    (-1,-1) -> 0
    (-1,0) -> 1
    (-1,1) -> 2
    (0,-1) -> 3
    (0,0) -> 4
    (0,1) -> 5
    (1,-1) -> 6
    (1,0) -> 7
    (1,1) -> 8
'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#3X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(3, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=[-1, 0, 1],   # previous state
    columns=[-1, 0, 1]  # current state
)

print(round(transition_table,2))

#3X3 RETURNS MATRIX
sums = df.groupby('pattern')['BZ=F'].mean().reindex(range(9), fill_value=0)
sums *= 100
matrix = sums.values.reshape(3, 3)
table_3x3 = pd.DataFrame(
    matrix,
    index=[-1, 0, 1],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_3x3,3))

/tmp/ipykernel_10924/4247617852.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

       -1      0      1
-1  31.27  34.98  33.75
 0  27.62  44.16  28.22
 1  31.93  39.49  28.58
       -1      0      1
-1 -2.770  0.088  2.525
 0 -2.198  0.068  2.096
 1 -2.429  0.077  2.427


# PART 2

In [10]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC','BTC-USD','BZ=F']
ticker = ['BZ=F']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['BZ=F'] < lower['BZ=F'],
    log_returns_categorized['BZ=F'] > upper['BZ=F']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
#This is different than lag of just 1, as there are two lags considered, total number of cases would be 27.
'''
    (-1,-1,-1) -> 0
    (-1,-1,0) -> 1
    (-1,-1,1) -> 2
    (-1,0,-1) -> 3
    (-1,0,0) -> 4
    (-1,0,1) -> 5
    (-1,1,-1) -> 6
    (-1,1,0) -> 7
    (-1,1,1) -> 8
    (0,-1,-1) -> 9
    (0,-1,0) -> 10
    (0,-1,1) -> 11
    (0,0,-1) -> 12
    (0,0,0) -> 13
    (0,0,1) -> 14
    (0,1,-1) -> 15
    (0,1,0) -> 16
    (0,1,1) -> 17
    (1,-1,-1) -> 18
    (1,-1,0) -> 19
    (1,-1,1) -> 20
    (1,0,-1) -> 21
    (1,0,0) -> 22
    (1,0,1) -> 23
    (1,1,-1) -> 24
    (1,1,0) -> 25
    (1,1,1) -> 26

'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(2) + 1) * 9 +
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#9X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(9, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous state
    columns=[-1, 0, 1]  # current state
)
print(round(transition_table,2))

#9X3 RETURNS MATRIX
sums = df.groupby('pattern')['BZ=F'].mean().reindex(range(27), fill_value=0)
sums *= 100
matrix = sums.values.reshape(9, 3)
table_9x3 = pd.DataFrame(
    matrix,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_9x3,3))

/tmp/ipykernel_10924/1143656851.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

          -1      0      1
-1,-1  30.00  32.79  37.21
-1,0   29.11  41.37  29.52
-1,1   35.99  32.97  31.03
0,-1   30.04  37.94  32.02
0,0    26.21  46.48  27.32
0,1    27.47  47.20  25.34
1,-1   33.94  33.71  32.35
1,0    28.41  43.17  28.41
1,1    32.82  37.15  30.03
          -1      0      1
-1,-1 -3.018  0.080  2.782
-1,0  -2.358  0.077  2.138
-1,1  -2.692  0.122  2.584
0,-1  -2.404  0.046  2.163
0,0   -1.999  0.070  1.951
0,1   -2.157  0.082  2.167
1,-1  -2.928  0.149  2.648
1,0   -2.325  0.055  2.265
1,1   -2.391  0.020  2.525




---



*GC=F*

GOLD

#PART 1

In [11]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC', 'BTC-USD','BZ=F','GC=F']
ticker = ['GC=F']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['GC=F'] < lower['GC=F'],
    log_returns_categorized['GC=F'] > upper['GC=F']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
'''
    (-1,-1) -> 0
    (-1,0) -> 1
    (-1,1) -> 2
    (0,-1) -> 3
    (0,0) -> 4
    (0,1) -> 5
    (1,-1) -> 6
    (1,0) -> 7
    (1,1) -> 8
'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#3X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(3, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=[-1, 0, 1],   # previous state
    columns=[-1, 0, 1]  # current state
)

print(round(transition_table,2))

#3X3 RETURNS MATRIX
sums = df.groupby('pattern')['GC=F'].mean().reindex(range(9), fill_value=0)
sums *= 100
matrix = sums.values.reshape(3, 3)
table_3x3 = pd.DataFrame(
    matrix,
    index=[-1, 0, 1],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_3x3,3))

/tmp/ipykernel_10924/3243043674.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

       -1      0      1
-1  27.34  39.36  33.29
 0  30.25  40.22  29.53
 1  32.35  40.29  27.36
       -1      0      1
-1 -1.295  0.068  1.220
 0 -1.113  0.063  1.160
 1 -1.087  0.074  1.258


#PART 2

In [12]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

#tickerlist = ['EURUSD=X','^GSPC','BTC-USD','BZ=F','GC=F']
ticker = ['GC=F']
end_date = dt.datetime(2025,12,31)
start_date = dt.datetime(2003,12,1)

#download all the data for the period, for the tickers listed
data = yf.download(ticker, start_date, end_date)

#prices as data downloaded has many columns Close, High, Low, Volumne
#log returns
prices = data['Close']
pct_change = prices.pct_change()
log_returns = np.log(1+pct_change)

#plotting the log_returns
#plt.plot(log_returns)

#defining the quantiles for different states
#<30% data in lower : State = -1
#30-70% data in middle : State = 0
#>70% data in upper : State = 1

lower = log_returns.quantile(0.30)
upper = log_returns.quantile(0.70)

#defining the categories
log_returns_categorized = log_returns.copy()
conditions = [
    log_returns_categorized['GC=F'] < lower['GC=F'],
    log_returns_categorized['GC=F'] > upper['GC=F']
]
choices = [-1, 1]
log_returns_categorized['category'] = np.select(conditions, choices, default=0)
log_returns_categorized = log_returns_categorized.dropna()

#labelling the categories
#This is different than lag of just 1, as there are two lags considered, total number of cases would be 27.
'''
    (-1,-1,-1) -> 0
    (-1,-1,0) -> 1
    (-1,-1,1) -> 2
    (-1,0,-1) -> 3
    (-1,0,0) -> 4
    (-1,0,1) -> 5
    (-1,1,-1) -> 6
    (-1,1,0) -> 7
    (-1,1,1) -> 8
    (0,-1,-1) -> 9
    (0,-1,0) -> 10
    (0,-1,1) -> 11
    (0,0,-1) -> 12
    (0,0,0) -> 13
    (0,0,1) -> 14
    (0,1,-1) -> 15
    (0,1,0) -> 16
    (0,1,1) -> 17
    (1,-1,-1) -> 18
    (1,-1,0) -> 19
    (1,-1,1) -> 20
    (1,0,-1) -> 21
    (1,0,0) -> 22
    (1,0,1) -> 23
    (1,1,-1) -> 24
    (1,1,0) -> 25
    (1,1,1) -> 26

'''

df = log_returns_categorized.copy()

df['pattern'] = (
    (df['category'].shift(2) + 1) * 9 +
    (df['category'].shift(1) + 1) * 3 +
    (df['category'] + 1)
)
df = df.dropna()

# days corresponding to each pattern
count = df['pattern'].value_counts().sort_index()
# probability attached with each pattern
probs = count / count.groupby(count.index // 3).transform('sum')
prob_states = probs*100

#9X3 PROBABILITY TRANSITION MATRIX
counts = count.sort_index().values.reshape(9, 3)
probs = counts / counts.sum(axis=1, keepdims=True)

transition_table = pd.DataFrame(
    probs*100,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous state
    columns=[-1, 0, 1]  # current state
)
print(round(transition_table,2))

#9X3 RETURNS MATRIX
sums = df.groupby('pattern')['GC=F'].mean().reindex(range(27), fill_value=0)
sums *= 100
matrix = sums.values.reshape(9, 3)
table_9x3 = pd.DataFrame(
    matrix,
    index=['-1,-1', '-1,0', '-1,1', '0,-1', '0,0', '0,1', '1,-1', '1,0', '1,1'],   # previous category
    columns=[-1, 0, 1]  # current category
)

print(round(table_9x3,3))

/tmp/ipykernel_10924/2809641444.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start_date, end_date)
[*********************100%***********************]  1 of 1 completed

          -1      0      1
-1,-1  26.59  36.26  37.14
-1,0   30.53  40.31  29.16
-1,1   32.73  35.44  31.83
0,-1   28.61  42.47  28.91
0,0    30.94  39.57  29.48
0,1    30.08  46.11  23.82
1,-1   26.39  38.10  35.50
1,0    29.10  40.90  30.00
1,1    35.16  37.80  27.03
          -1      0      1
-1,-1 -1.412  0.077  1.250
-1,0  -1.194  0.074  1.184
-1,1  -1.126  0.082  1.314
0,-1  -1.186  0.059  1.180
0,0   -1.005  0.075  1.126
0,1   -1.060  0.056  1.158
1,-1  -1.344  0.073  1.234
1,0   -1.182  0.037  1.183
1,1   -1.076  0.096  1.307
